In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets evaluate scikit-learn accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [53]:
import pandas as pd

file_path = "/content/drive/MyDrive/plant_training/text_corpus1.csv"

df = pd.read_csv(file_path)

print("Rows:", len(df))
print("Classes:", df["class_name"].nunique())
df.head()

Rows: 3960
Classes: 58


,class_name,text
0,Apple_Apple_Scab,Field observation in Apple reveals Olive to da...
1,Apple_Apple_Scab,Apple leaf condition includes Olive to dark-br...
2,Apple_Apple_Scab,The Apple plant leaf shows Olive to dark-brown...
3,Apple_Apple_Scab,Symptoms in Apple crop include Olive to dark-b...
4,Apple_Apple_Scab,Apple plants display Olive to dark-brown scab ...


In [54]:
def remove_crop_name(row):
    crop = row["class_name"].split("_")[0].lower()
    text = row["text"].lower()
    return text.replace(crop, "")

df["text"] = df.apply(remove_crop_name, axis=1)

In [55]:
import random

def advanced_augment(text):

    variations = []

    # 1. Original
    variations.append(text)

    # 2. Lowercase
    variations.append(text.lower())

    # 3. Short version
    short = " ".join(text.split()[:8])
    variations.append(short)

    # 4. Add noise phrases
    noise = random.choice([
        "seen in field",
        "observed recently",
        "spreading slowly",
        "worsening condition",
        "initial stage",
        "severe infection"
    ])
    variations.append(text + " " + noise)

    # 5. Synonym swaps
    t = text.replace("spots", "lesions")
    t = t.replace("yellow", "yellowish")
    t = t.replace("brown", "dark brown")
    t = t.replace("leaf", "plant leaves")
    variations.append(t)

    # 6. Reordered sentence
    words = text.split()
    if len(words) > 6:
        random.shuffle(words)
        variations.append(" ".join(words))

    return random.choice(variations)

In [56]:
augmented_rows = []

for _, row in train_base.iterrows():
    for _ in range(5):   # 🔥 increase to 5x data
        new_text = advanced_augment(row["text"])

        augmented_rows.append({
            "class_name": row["class_name"],
            "text": new_text
        })

train_df = pd.DataFrame(augmented_rows)

print("Augmented train size:", train_df.shape)

Augmented train size: (3445, 2)


In [57]:
train_df = train_df.drop_duplicates(subset=["text"]).reset_index(drop=True)
val_df = val_df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print("Final train:", train_df.shape)
print("Final val:", val_df.shape)

Final train: (2706, 2)
Final val: (173, 3)


In [58]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

train_df["label"] = le.fit_transform(train_df["class_name"])
val_df["label"] = le.transform(val_df["class_name"])

num_labels = len(le.classes_)
print("Classes:", num_labels)

Classes: 58


In [59]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])

In [60]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])

In [61]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/2706 [00:00<?, ? examples/s]

Map:   0%|          | 0/173 [00:00<?, ? examples/s]

In [62]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [64]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",        # ✅ changed
    save_strategy="epoch",
    logging_strategy="epoch",     # (good to add)

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,
    learning_rate=3e-5,
    weight_decay=0.05,

    load_best_model_at_end=True,
)

In [65]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

In [68]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,3.428768,2.216053,0.838150,0.800896
2,1.634894,0.845691,1.000000,1.000000
3,0.695430,0.362285,1.000000,1.000000
4,0.352211,0.200808,1.000000,1.000000
5,0.239903,0.163923,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=850, training_loss=1.2702411651611327, metrics={'train_runtime': 384.9869, 'train_samples_per_second': 35.144, 'train_steps_per_second': 2.208, 'total_flos': 890420624501760.0, 'train_loss': 1.2702411651611327, 'epoch': 5.0})